# CEE 490 — Finite-Difference Site Response: Stages 1–3

This notebook runs the current implementation from `src/site_response/`. The notebook is presentation-focused: solver logic lives in `.py` files, while this notebook organizes the theory, checks, plots, and verification outputs.


## Governing equations

The 1D SH-wave velocity-stress system is

$$\rho(z)\frac{\partial v}{\partial t}=\frac{\partial \tau}{\partial z}$$

$$\frac{\partial \tau}{\partial t}=\mu(z)\frac{\partial v}{\partial z}$$

where $v(z,t)$ is horizontal particle velocity, $\tau(z,t)$ is shear stress, and $\mu=\rho V_s^2$.


In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
SRC_DIR = PROJECT_ROOT / 'src'
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from site_response.analysis import run_homogeneous_verification_case
from site_response.materials import homogeneous_rock_model, layered_soil_over_rock_model, material_summary
from site_response.parameters import default_config, format_parameter_summary, compute_parameter_checks
from site_response.source import build_source_time_function, normalized_amplitude_spectrum
from site_response.utils import prepare_project_outputs, save_simulation_result_npz
from site_response.visualization import (
    apply_project_style, plot_velocity_profile, plot_impedance_profile,
    plot_source_time_history, plot_source_spectrum, plot_homogeneous_seismogram,
    plot_wavefield_image,
)

apply_project_style()


## Stage 1 — Project setup and numerical checks

The explicit solver must satisfy the Courant stability condition

$$C=\frac{V_{s,\max}\Delta t}{\Delta z}\leq1$$

and the grid should resolve the shortest interpreted wavelength:

$$N_{\mathrm{ppw}}=\frac{V_{s,\min}/f_{\max}}{\Delta z}. $$


In [ ]:
config = default_config()
prepare_project_outputs(config)
print(format_parameter_summary(config))
compute_parameter_checks(config)


## Stage 2 — Material models and source

Material stiffness is computed from

$$\mu(z)=\rho(z)V_s^2(z).$$

The source is a Ricker wavelet:

$$s(t)=\left[1-2\pi^2 f_0^2(t-t_0)^2\right]\exp\left[-\pi^2 f_0^2(t-t_0)^2\right].$$


In [ ]:
homogeneous_material = homogeneous_rock_model(config)
layered_material = layered_soil_over_rock_model(config)
print(material_summary(homogeneous_material))
print(material_summary(layered_material))


In [ ]:
plot_velocity_profile(layered_material, output_path=config.figures_dir / 'velocity_profile.png')


In [ ]:
plot_impedance_profile(layered_material, output_path=config.figures_dir / 'impedance_profile.png')


In [ ]:
time_s, source_values = build_source_time_function(config)
plot_source_time_history(time_s, source_values, output_path=config.figures_dir / 'source_wavelet.png')


In [ ]:
frequencies_hz, source_spectrum = normalized_amplitude_spectrum(time_s, source_values)
plot_source_spectrum(frequencies_hz, source_spectrum, config.dominant_frequency_hz, output_path=config.figures_dir / 'source_spectrum.png')


## Stage 3 — Homogeneous solver verification

For a homogeneous material, the theoretical travel time from source to receiver is

$$t_{\mathrm{travel}}=\frac{d}{V_s}.$$

Because the Ricker wavelet peaks at $t_0$, the expected peak arrival is

$$t_{\mathrm{peak,theory}}=t_0+\frac{d}{V_s}.$$

The percent error is

$$\mathrm{Error}(\%)=\frac{|t_{\mathrm{num}}-t_{\mathrm{theory}}|}{t_{\mathrm{theory}}}\times100.$$


In [ ]:
homogeneous_result, arrival_check = run_homogeneous_verification_case(config)
arrival_table = arrival_check.as_dataframe()
arrival_table


In [ ]:
save_simulation_result_npz(homogeneous_result, config.results_dir / 'homogeneous_results.npz')
arrival_table.to_csv(config.results_dir / 'homogeneous_arrival_check.csv', index=False)
plot_homogeneous_seismogram(homogeneous_result, arrival_check, output_path=config.figures_dir / 'homogeneous_verification.png')


In [ ]:
plot_wavefield_image(homogeneous_result, output_path=config.figures_dir / 'homogeneous_wavefield.png')


## Notes for the next stage

The homogeneous solver is now ready for grid-refinement testing and then layered-site response. The layered material model and source plots are already available, but amplification analysis should be added only after the homogeneous verification behaves acceptably.
